# LLMs in an agent harness for long-horizon tasks

An agent is a language model that works over many turns. It writes something, a tool runs it, the tool's answer comes back as text, and the model writes again. A coding agent edits a file, runs the tests and reads the failures; a browsing agent clicks, reads the page and clicks again. Only at the end does anyone say how well the whole thing went.

This notebook trains such a loop with reinforcement learning, small enough to run on a CPU. On the way it shows what every agent trainer has to get right: how a multi-turn session is recorded, how tool calls run in a sandbox, how a trajectory becomes tokens and a loss mask, with the model's tokens trained and the tool's not, and how one reward at the end of a long episode reaches every token the model wrote. It builds on [SFT for LLMs](09-sft-for-llms.ipynb), where the loss mask comes from, and on [Reinforcement learning with verifiable rewards](10-rl-with-verifiable-rewards.ipynb), which derives the GRPO objective used here.

**Runs on a CPU in a few minutes.** The model is tiny and starts from random weights, so it learns one simple skill, not language. The machinery is the one a real harness uses with a pretrained model; the last section points to that.

In [ ]:
%pip install -q "dewml @ git+https://github.com/AshishKumar4/dew"

## Settings

A task asks the agent to bring a counter to a goal. The agent has `TURNS` turns. In each it writes up to `ACTION_TOKENS` characters and ends its action; a tool then adds one to the counter for every `+` in the action and takes one away for every `-`, and reports the new value. `GROUP` attempts at each task are compared with one another, which is how GRPO judges an attempt.

In [1]:
import os

TURNS = 4            # tool calls in an episode
ACTION_TOKENS = 8    # the most tokens one action may hold, its end token included
WIDTH = 64           # the most tokens one episode's trajectory may hold
TASKS = 8            # tasks in a training step
GROUP = 4            # attempts at each task
STEPS = 60
LEARNING_RATE = 1e-2
SEED = 0

# The tutorial test sets DEW_TUTORIAL_SMOKE=1 to run every cell in minutes on a CPU.
if os.environ.get('DEW_TUTORIAL_SMOKE') == '1':
    TASKS, STEPS = 4, 3

## A tokenizer

A real agent reads and writes text through its tokenizer, and the trainer only ever sees token ids. Ours has one token per character of a small alphabet, plus an end-of-action token, `END`, which the model writes to hand the turn to the tool. Below, `END` prints as `|` and a newline as `↵`.

In [2]:
import numpy as np

ALPHABET = '0123456789+-= \ngoal'
END = len(ALPHABET)


def encode(text: str) -> list[int]:
    return [ALPHABET.index(character) for character in text]


def decode(ids) -> str:
    return ''.join('|' if token == END else ALPHABET[token] for token in ids)


def shown(ids) -> str:
    return decode(ids).replace('\n', '↵')


prompt = encode('goal 7\n=0\n')
print(prompt, shown(prompt))

[15, 16, 17, 18, 13, 7, 14, 12, 0, 14] goal 7↵=0↵


## The tool runs in a sandbox

The agent's actions are untrusted text, so the harness never runs them inside the training process. `dew.rl.sandbox` runs a `Program`: files written to a fresh temporary directory and a command run there, with no shell, under CPU, memory and time limits. It reports an `Outcome` with the exit status and the output. A `SandboxFleet` runs many programs at once on worker threads, each in its own process.

Our tool is a three-line Python program. It reads the counter and an action on stdin and prints the new counter.

`ProcessRunner` limits resources but shares your user, filesystem and network. For code you don't trust at all, `ContainerRunner` runs each program in a fresh container with no network.

In [3]:
from dew.rl.sandbox import Program, ProcessRunner, SandboxFleet, SandboxLimits

COUNTER = '''
import sys
state, action = sys.stdin.read().split("\\n", 1)
print(int(state) + action.count("+") - action.count("-"))
'''

runner = ProcessRunner()
fleet = SandboxFleet(runner, limits=SandboxLimits(wall_seconds=10, cpu_seconds=5), workers=TASKS * GROUP)


def counter(state: int, action: str) -> int:
    program = Program({'counter.py': COUNTER}, (*runner.python, 'counter.py'), stdin=f'{state}\n{action}')
    return int(fleet.submit(program).result().stdout)


print(counter(0, '++3-+'))

2


## The environment

The harness's side of an episode is an `Environment` with two methods. `reset` returns the first context the model reads: the goal and the counter. `step` takes the model's action, runs it through the tool and returns the next context: the old context, then the action, then the tool's answer. The context only ever grows, so on every turn the model sees the whole episode so far. After `TURNS` actions the environment says the episode is complete.

The verifier scores a finished episode once: $R = 1 - |g - s| / g$, where $g$ is the goal and $s$ the counter's last value. It is 1 when the counter lands on the goal and falls off on either side. It reads $s$ from the trajectory itself, the last thing the tool said.

In [4]:
from dew.objectives.rl import EpisodeStatus, Observation


def goal_of(task) -> int:
    return 3 + int(task.id) % 5


class Counter:
    def __init__(self, goal: int):
        self.goal, self.state, self.turns = goal, 0, 0

    def reset(self) -> Observation:
        return Observation(tuple(encode(f'goal {self.goal}\n={self.state}\n')))

    def step(self, action) -> Observation:
        self.turns += 1
        self.state = counter(self.state, decode(action.tokens[:-1]))  # the action without its END
        context = action.context + action.tokens + tuple(encode(f'={self.state}\n'))
        done = self.turns == TURNS
        return Observation(context, EpisodeStatus.COMPLETED if done else EpisodeStatus.RUNNING)


def last_state(episode) -> int:
    contexts = [turn.observation.context for turn in episode.transitions if turn.observation.context]
    return int(decode(contexts[-1] if contexts else episode.initial.context).rsplit('=', 1)[1])


def verify(task, episode) -> float:
    goal = goal_of(task)
    return 1 - abs(goal - last_state(episode)) / goal

## A model and its server

The policy $\pi_\theta$ is a two-layer `CausalTransformer` over our 20 tokens, from random weights. Rollouts come from Dew's own inference server, `NativeRolloutServer`, which decodes many episodes' calls together and, during training, serves the newest weights after each update.

An `EnvironmentSource` runs each episode in a thread of its own: reset the environment, ask the server for an action, step, and again, until the environment says the episode is over or a budget runs out. Then it calls the verifier.

In [5]:
from contextlib import contextmanager

import jax

from dew.inference import NativeRolloutServer, TextGeneration
from dew.inference.serving import Server
from dew.nn.backbones.causal_transformer import CausalTransformer
from dew.objectives.rl import EnvironmentSource, GRPOObjective
from dew.sampling import Sampling

print(jax.devices())
model = CausalTransformer(vocab_size=END + 1, emb_features=32, num_layers=2, num_heads=2, head_dim=16,
                          mlp_features=64, max_seq_len=WIDTH, dtype='float32')
variables = GRPOObjective(model, WIDTH - 1).init(jax.random.key(SEED))
# The objective trains from the weights the server starts with.
objective = GRPOObjective(model, WIDTH - 1, variables=variables)
server = NativeRolloutServer(Server.from_task(
    TextGeneration(model, variables, None, sampling=Sampling(temperature=1.0, eos_token_ids=END)),
    slots=TASKS * GROUP, capacity=WIDTH))


@contextmanager
def environment(task, identity):
    yield Counter(goal_of(task))


source = EnvironmentSource(server, environment, verify, max_prompt_tokens=WIDTH - ACTION_TOKENS,
                           max_new_tokens=ACTION_TOKENS, max_turns=TURNS, workers=TASKS * GROUP)

[CpuDevice(id=0)]


## One group of episodes

Before training, run a group by hand. `submit` starts `GROUP` episodes of one task and returns a future for each `Session`, the record a trainer learns from. A session holds its calls in order. Each call is exactly the token ids the model read (`prompt_ids`) and wrote (`sampled_ids`), with the log-probability of each written token under the policy that sampled it. The tool's answers have no field of their own: they are in the next call's prompt, because that is what the model read.

An action that runs out of `ACTION_TOKENS` before its `END` is cut off, and its episode ends as truncated: a harness never runs half an action.

In [6]:
from dew.objectives.rl import Task

sessions = [future.result() for future in source.submit(Task('2'), GROUP, version=0, key=jax.random.key(SEED))]
for session in sessions:
    print(f'{session.status.value:>9}  reward {session.reward:+.2f}  calls {len(session.calls)}')
for number, call in enumerate(sessions[0].calls):
    print(f'call {number}: read {shown(call.prompt_ids)!r:44} wrote {shown(call.sampled_ids)!r}')

truncated  reward +0.00  calls 1
truncated  reward -0.20  calls 4
truncated  reward +0.00  calls 1
truncated  reward +0.00  calls 1
call 0: read 'goal 5↵=0↵'                                 wrote '6=4↵366='


## From a trajectory to tokens

`pack` turns sessions into the arrays the loss reads. When each call's prompt extends the call before it, as ours do, a session's calls merge into one chain of tokens: the whole episode is one sequence, its context never repeated. Chains are packed into rows of `WIDTH` tokens, and beside the ids come, per token:

- `response_mask`, $m_t$: 1 on the tokens the model sampled, its `END`s included, and 0 on the prompt, the tool's answers and padding;
- `behavior_log_probs`: $\log \pi_{\theta_\text{old}}(x_t \mid x_{<t})$ of each sampled token, as the sampler drew it;
- `advantages`, $A$: the episode's advantage, the same on all its tokens.

Here is the first episode, one character a column, with `^` under the tokens the loss trains.

In [7]:
from dew.objectives.rl import pack

batch = pack(sessions, WIDTH, truncation='score')
print({name: array.shape for name, array in batch.items()})
first = batch['session_index'] == 0
print(shown(batch['input_ids'][first]))
print(''.join('^' if trained else ' ' for trained in batch['response_mask'][first]))

{'input_ids': (2, 64), 'text_segment_ids': (2, 64), 'text_positions': (2, 64), 'response_mask': (2, 64), 'behavior_log_probs': (2, 64), 'versions': (2, 64), 'session_index': (2, 64), 'call_index': (2, 64), 'advantages': (2, 64), 'session_weights': (2, 64)}
goal 5↵=0↵6=4↵366=
          ^^^^^^^^


## One reward for the whole episode

The verifier speaks once, at the end, about everything the agent did. GRPO turns the group's rewards into advantages,

$$A^{(i)} = \frac{R^{(i)} - \operatorname{mean}_j R^{(j)}}{\operatorname{std}_j R^{(j)} + \varepsilon},$$

and every token episode $i$ sampled, on every turn, gets the same $A^{(i)}$. No turn is singled out: a `+` written on the first turn and one written on the last get the same credit. Over many episodes, what tends to come before a good ending gains probability and what comes before a bad one loses it. That is the price of a reward at the end: credit assignment is left to averaging, which is why long-horizon training wants many episodes and a reward that varies within a group.

In [8]:
rewards = np.array([session.reward for session in sessions])
print('rewards   ', np.round(rewards, 3))
for number in range(GROUP):
    trained = (batch['session_index'] == number) & (batch['response_mask'] > 0)
    print(f'episode {number}: advantage {np.unique(batch["advantages"][trained]).round(3)} '
          f'on {int(trained.sum())} sampled tokens')

rewards    [ 0.  -0.2  0.   0. ]
episode 0: advantage [0.5] on 8 sampled tokens
episode 1: advantage [-1.5] on 24 sampled tokens
episode 2: advantage [0.5] on 8 sampled tokens
episode 3: advantage [0.5] on 8 sampled tokens


## The gradient flows only through sampled tokens

The policy loss is a mean over sampled tokens: every term carries $m_t$, and the mean is over $\sum_t m_t$. The prompt and the tool's answers still matter, since the model reads them as context for every token it writes after them. But nothing pushes their own probability up or down: the model is not trained to predict what the tool says.

Both halves can be checked. Putting a huge advantage on the tokens the model only read leaves the gradient exactly as it was. Changing what the tool said changes the gradient, because the tokens the model wrote afterwards were conditioned on it.

In [9]:
import jax.numpy as jnp

from dew.objectives.base import Step

step = Step(step=jnp.asarray(0), key=jax.random.key(0), ema=None)


def gradient(batch):
    return jax.grad(lambda variables: objective.loss(variables, batch, step)[0].mean()[0])(variables)


def same(left, right) -> bool:
    return all(np.array_equal(a, b) for a, b in zip(jax.tree.leaves(left), jax.tree.leaves(right), strict=True))


read = (batch['response_mask'] == 0) & (batch['text_segment_ids'] > 0)
louder = {**batch, 'advantages': np.where(read, 100.0, batch['advantages']).astype(np.float32)}
digits = read & (batch['input_ids'] < 10)
rewritten = {**batch, 'input_ids': np.where(digits, (batch['input_ids'] + 1) % 10, batch['input_ids'])}
reference = gradient(batch)
print('a huge advantage on the tokens it read, the same gradient:', same(reference, gradient(louder)))
print('the tool said something else, the same gradient:          ', same(reference, gradient(rewritten)))

a huge advantage on the tokens it read, the same gradient: True


the tool said something else, the same gradient:           False


## Training

A `RolloutScheduler` connects the source to the `Trainer`. For each step it takes `TASKS` task ids from the data, starts `GROUP` episodes of each, waits for whole groups, packs them and hands the batch to the trainer; after the update it publishes the new weights to the server.

Two settings matter here. `max_lag=0` and `ahead=0` sample every batch with the weights it trains, which keeps the update exactly on-policy. A real run lets rollouts run ahead of the trainer, so a GPU never waits on a slow episode, and corrects for the lag, which notebook 10 explains. And `truncation='score'` trains an episode whose action ran out of tokens on the reward of what it reached, rather than dropping it: a random model rarely ends four actions in time, and the default, `'mask'`, would leave it nothing to learn from.

In [10]:
import itertools

import optax

from dew.data.dataset import Loader
from dew.objectives.rl import RolloutScheduler
from dew.training import Trainer

records = []
scheduler = RolloutScheduler(objective, source, server, width=WIDTH, rows=TASKS * GROUP, groups=GROUP,
                             max_lag=0, ahead=0, truncation='score', log=records.append)
ids = ({'task_id': np.arange(TASKS, dtype=np.int32) + step * TASKS} for step in itertools.count())
tasks = scheduler.tasks(Loader(train=lambda partition: ids, val=None, records=None, batch=TASKS))
trainer = Trainer(objective, optax.adam(LEARNING_RATE), key=jax.random.key(SEED), rollout=scheduler)
state = trainer.fit(tasks, steps=STEPS, log_every=STEPS)

Training CausalTransformer from step 0 to 60: 21,344 parameters, on 1 × cpu, batch 8, float32

step 60/60  loss -0.008868  actor/pg_clipfrac 0  actor/pg_clipfrac_lower 0  actor/ppo_kl 0  mismatch/ess 1.000  mismatch/k3_kl 0  mismatch/kl 1.799e-09  pg -0.008868  data_wait_ms 0.03750  grad_norm 0.2073  compiles 1.000  cache_hits 0  cache_misses 0  step_time_ms 827.0  samples_per_sec 38.69  rollout_seconds 52.70  accepted 100.0%  groups 8.000  lag 0  lag/max 0  lag/mean 0  latency/max 1.114  latency/p50 1.021  latency/p90 1.095  latency/p99 1.111  masked/cancelled 0  masked/infra_error 0  merge/calls_per_chain 4.000  pack/fill 42.6%  resubmitted 0  reward/mean 0.7189  status/agent_error 0.0%  status/cancelled 0.0%  status/completed 100.0%  status/infra_error 0.0%  status/truncated 0.0%  waited_seconds 1.071

Trained 60 steps in 0:00:56: first step after 6.31 s, then 1.2 step/s

88.7% of the wall time in steps, final loss -0.008868

The mean reward over each step's episodes, and the share that ended every action in time:

In [11]:
every = max(STEPS // 10, 1)
for record in records[::every]:
    metrics = record.metrics
    print(f'step {record.updates:3d}  reward {metrics["reward/mean"]:+.3f}  '
          f'completed {metrics.get("status/completed", 0.0):.0%}')

step   0  reward +0.005  completed 3%
step   6  reward -0.005  completed 0%
step  12  reward +0.000  completed 0%
step  18  reward +0.000  completed 0%
step  24  reward +0.267  completed 78%
step  30  reward +0.628  completed 97%
step  36  reward +0.716  completed 100%
step  42  reward +0.652  completed 100%
step  48  reward +0.530  completed 97%
step  54  reward +0.717  completed 100%


## After training

Two new groups, sampled by the trained weights, which the scheduler published to the server, at a goal of 4 and a goal of 7. Each line is the last call's context and action: the whole episode but the tool's final answer.

The policy has learnt to end each action in time and to put a `+` in it: one per turn, which brings the counter to 4 in four turns. It has not learnt to read the goal, which would mean writing more `+` for a larger one. That takes many more steps, and a reward that tells the two apart within a group; a pretrained model, which already reads numbers, would start much closer.

In [12]:
for task in (Task('1'), Task('4')):
    futures = source.submit(task, GROUP, version=server.version, key=jax.random.key(SEED + 1))
    for session in (future.result() for future in futures):
        last = session.calls[-1]
        print(f'{session.status.value:>9}  reward {session.reward:+.2f}  {shown(last.prompt_ids + last.sampled_ids)}')
scheduler.close()
source.close()
server.close()
fleet.close()

completed  reward +1.00  goal 4↵=0↵+|=1↵+|=2↵+|=3↵+|
completed  reward +0.75  goal 4↵=0↵+|=1↵+|=2↵+8+|=4↵+|


completed  reward +0.75  goal 4↵=0↵+|=1↵+↵+|=3↵+|=4↵+|
completed  reward +1.00  goal 4↵=0↵+|=1↵+|=2↵+|=3↵+|
completed  reward +0.57  goal 7↵=0↵+|=1↵+|=2↵+|=3↵+|
completed  reward +0.71  goal 7↵=0↵+|=1↵+|=2↵+8+|=4↵+|


completed  reward +0.71  goal 7↵=0↵+|=1↵+↵+|=3↵+|=4↵+|
completed  reward +0.57  goal 7↵=0↵+|=1↵+|=2↵+|=3↵+|


## Real harnesses

Everything above ran in one process, with the environment as a Python object. Production agents run in a harness of their own, a coding agent in a container, say, that calls the model over HTTP and decides for itself what to send it. [`examples/train_harbor.py`](https://github.com/AshishKumar4/dew/blob/main/examples/train_harbor.py) trains through one. Harbor starts each task's sandbox and agent, a gateway in front of the inference engines records every call the agent makes as a session call, the same scheduler groups and packs them, and GRPO trains on exactly the tokens the model sampled, as here. `python examples/train_harbor.py --smoke` runs it on a CPU with a tiny model; [`examples/train_rlvr.py`](https://github.com/AshishKumar4/dew/blob/main/examples/train_rlvr.py) shows a text harness of its own, where the model writes Python, the sandbox runs it against tests, and a failing attempt hears how many passed and tries again.